# 10 — Train Text Model

Fine-tunes every model listed in `config.MODELS_TO_RUN` on the MELD training
set and saves the best checkpoint (by dev macro-F1) for each.

**Depends on:** `01_preprocess_text.ipynb` (run automatically below).

**Outputs** (written to `../experiments/text_unimodal/<version>/<model_slug>/`):
- `model/`               — best checkpoint (HuggingFace format)
- `history.csv`          — per-epoch train/dev metrics
- `config.json`          — full run config
- `train_metrics.json`   — training summary (params, time, power, best dev F1)
- `power_samples.csv`    — raw GPU power readings

In [ ]:
# Run the preprocessing notebook — this sets train_df, dev_df, test_df,
# label2id, id2label, labels_sorted in this kernel's namespace.
%run 01_preprocess_text.ipynb

In [ ]:
from src.config import MODELS_TO_RUN, DEVICE
from src.training.train import train_one

print('Device:', DEVICE)
print('Models to train:', MODELS_TO_RUN)

## Train

Each call to `train_one` runs the full training loop and returns the path to
the run directory. Bump `config.VERSION` if you want a clean slate.

In [ ]:
run_dirs = []

for model_name in MODELS_TO_RUN:
    run_dir = train_one(
        model_name=model_name,
        train_df=train_df,
        dev_df=dev_df,
        test_df=test_df,
        label2id=label2id,
        id2label=id2label,
    )
    run_dirs.append(run_dir)

print('\nAll run directories:')
for d in run_dirs:
    print(' ', d)

## Quick peek at training history

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(run_dirs), figsize=(7 * len(run_dirs), 4), squeeze=False)

for ax, run_dir in zip(axes[0], run_dirs):
    history_path = os.path.join(run_dir, 'history.csv')
    if not os.path.exists(history_path):
        continue
    hist = pd.read_csv(history_path)
    ax.plot(hist['epoch'], hist['train_loss'],    label='train loss',    marker='o')
    ax.plot(hist['epoch'], hist['dev_loss'],      label='dev loss',      marker='s')
    ax.plot(hist['epoch'], hist['dev_macro_f1'],  label='dev macro-F1',  marker='^')
    ax.set_xlabel('Epoch')
    ax.set_title(os.path.basename(run_dir).replace('_', '-'))
    ax.legend()

plt.suptitle('Training history', fontsize=13)
plt.tight_layout()
plt.show()